# Project 2 — Phase 3: Train + Deploy the Code-Switching Language ID Model
**Code Switching NLP | Code Saviours SI-26 | Humna Imran**

*Environment: Google Colab (GPU runtime) — repo cloned into Google Drive*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hamnasz/code-switching-codesaviours-si26-humna/blob/main/SI26-Week7/SI26-Week7-Humna.ipynb)
<!-- ^ this link only resolves once SI26-Week7-Humna.ipynb is actually pushed to that path in the repo (Step 8) -->

Week 6 built `dataset.csv` — a labeled Roman Urdu + English code-switching dataset (`sentence`, `word`, `label`
with `label ∈ {URD, ENG, MIX}`). This notebook fine-tunes a token-classification model (`xlm-roberta-base`) so
it can label unseen Roman Urdu text word-by-word, publishes it to the Hugging Face Hub, and wraps it in a
small Streamlit demo.

**How this notebook is organized for Colab:**
- Your GitHub repo is **cloned straight into Google Drive** (Step 0), so `dataset.csv`, this notebook, and
  everything you write (`app.py`, `eval_metrics.json`) live in one place you can `git push` from.
- **Training checkpoints are also saved to Drive**, but in a *separate* folder outside the git repo — model
  checkpoints are gigabyte-scale binary files, and committing those to GitHub would blow past its 100 MB
  per-file limit. If Colab disconnects mid-training, just re-run the notebook from the top: it detects the
  latest checkpoint in Drive and resumes instead of starting over.
- The *actual* deliverable copy of your trained model is the one you push to the Hugging Face Hub in Step 5 —
  Drive checkpoints exist only so you don't lose training progress to a Colab timeout.

**Before you start:** `Runtime > Change runtime type > T4 GPU` (or better, if available on your account).


## Step 0a — Confirm the GPU runtime is on

In [ ]:
import torch

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('\n⚠️  No GPU detected.')
    print('Go to Runtime > Change runtime type > Hardware accelerator > GPU (T4),')
    print('then Runtime > Restart session, then re-run this notebook from the top.')


## Step 0b — Mount Drive and clone the repo into it

Everything below lives under Google Drive so it survives Colab disconnects. If the repo is already cloned
in your Drive from a previous session, this just `git pull`s the latest instead of re-cloning.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive'
REPO_NAME = 'code-switching-codesaviours-si26-humna'
REPO_ROOT = f'{DRIVE_ROOT}/{REPO_NAME}'

GITHUB_REPO_URL = 'https://github.com/hamnasz/code-switching-codesaviours-si26-humna.git'


In [ ]:
from getpass import getpass
import os

# Your repo is public, so you can just press Enter here — no token needed.
# (Kept as a prompt anyway so this still works unchanged if you ever make the repo private.)
# Using getpass keeps the token out of the notebook's saved output/history.
GITHUB_TOKEN = getpass('GitHub personal access token (blank if repo is public): ')

clone_url = GITHUB_REPO_URL
if GITHUB_TOKEN:
    clone_url = GITHUB_REPO_URL.replace('https://', f'https://{GITHUB_TOKEN}@')

if not os.path.isdir(REPO_ROOT):
    print(f'Cloning into {REPO_ROOT} ...')
    !git clone "{clone_url}" "{REPO_ROOT}"
else:
    print(f'Repo already present at {REPO_ROOT} — pulling latest changes ...')
    !git -C "{REPO_ROOT}" pull

WEEK7_DIR = f'{REPO_ROOT}/SI26-Week7'
os.makedirs(WEEK7_DIR, exist_ok=True)

DATASET_PATH = f'{REPO_ROOT}/SI26-Week6/dataset.csv'
print('\nREPO_ROOT   =', REPO_ROOT)
print('WEEK7_DIR   =', WEEK7_DIR)
print('DATASET_PATH=', DATASET_PATH)


> **This notebook file itself should live inside `WEEK7_DIR`** so it's part of the git repo you push in
> Step 8. Easiest path: after cloning above, drag this `.ipynb` into the `SI26-Week7` folder in the Drive web
> UI (or `File > Save a copy in Drive` to that folder), then reopen it from there so Colab's autosave keeps
> writing to the right place.

## Step 0c — Install packages

Colab already ships `torch` pre-matched to its CUDA driver, so we don't touch it — reinstalling via pip can
break that match. Everything else installs quickly.

In [ ]:
%pip install -q transformers datasets scikit-learn huggingface_hub

import json
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score

print('Torch version:', torch.__version__)


## Step 1 — Load `dataset.csv` and group into sentences

In [ ]:
df = pd.read_csv(DATASET_PATH)
assert {'sentence', 'word', 'label'}.issubset(df.columns), \
    f'Expected sentence/word/label columns, got: {list(df.columns)}'

label_list = ['URD', 'ENG', 'MIX']
label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for i, l in enumerate(label_list)}

bad_labels = set(df['label'].unique()) - set(label_list)
assert not bad_labels, f'Found unexpected labels in dataset.csv: {bad_labels}'

missing_labels = set(label_list) - set(df['label'].unique())
if missing_labels:
    print(f'⚠️  No examples at all for: {sorted(missing_labels)} — the model cannot learn these classes, '
          f'and their F1 will be 0.0 in evaluation. See the note above Step 3 for why.')

# Group word-by-word rows back into per-sentence word/label sequences,
# preserving original word order (no sort=True, which would scramble sentences)
grouped = df.groupby('sentence', sort=False)
sentences = [
    {'words': g['word'].astype(str).tolist(), 'labels': g['label'].tolist()}
    for _, g in grouped
]

print(f'Loaded {len(df)} word rows across {len(sentences)} sentences')
print('Label distribution (word level):')
print(df['label'].value_counts())

train_data, test_data = train_test_split(sentences, test_size=0.2, random_state=42)
print(f'\nTraining sentences: {len(train_data)}')
print(f'Testing sentences:  {len(test_data)}')


## Step 2 — Tokenize and align labels to sub-word tokens

XLM-RoBERTa splits words into sub-word pieces, so each word's label has to be copied onto only its *first*
sub-token (`-100` everywhere else — the Trainer's loss function ignores `-100`).

In [ ]:
from transformers import (AutoTokenizer, AutoModelForTokenClassification,
                          TrainingArguments, Trainer, DataCollatorForTokenClassification)
from datasets import Dataset

model_name = 'xlm-roberta-base'
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_and_align_labels(examples):
    tokenized = tokenizer(examples['words'], truncation=True, is_split_into_words=True)
    all_labels = []
    for i, label in enumerate(examples['labels']):
        word_ids = tokenized.word_ids(batch_index=i)
        label_ids, prev_word = [], None
        for word_id in word_ids:
            if word_id is None:
                label_ids.append(-100)
            elif word_id != prev_word:
                label_ids.append(label2id[label[word_id]])
            else:
                label_ids.append(-100)
            prev_word = word_id
        all_labels.append(label_ids)
    tokenized['labels'] = all_labels
    return tokenized

def to_hf_dataset(data):
    return Dataset.from_dict({
        'words': [d['words'] for d in data],
        'labels': [d['labels'] for d in data],
    })

train_ds = to_hf_dataset(train_data).map(tokenize_and_align_labels, batched=True)
test_ds = to_hf_dataset(test_data).map(tokenize_and_align_labels, batched=True)

print(train_ds)
print(test_ds)


## Step 3 — Fine-tune XLM-RoBERTa, checkpointing to Drive

`compute_metrics` reports **precision / recall / F1 per label (URD, ENG, MIX)** plus a macro-F1 — exactly
what the submission asks you to paste into the Classroom comment.

Checkpoints go to `CHECKPOINT_DIR` in Drive, **not** inside the git repo — see the note at the top of the
notebook for why. `save_total_limit=2` keeps only the 2 most recent checkpoints so Drive doesn't fill up.

In [ ]:
model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id,
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=2)

    true_labels, true_preds = [], []
    for pred_row, label_row in zip(predictions, labels):
        for p, l in zip(pred_row, label_row):
            if l != -100:
                true_labels.append(id2label[l])
                true_preds.append(id2label[p])

    report = classification_report(
        true_labels, true_preds, labels=label_list, output_dict=True, zero_division=0
    )
    metrics = {
        'accuracy': report['accuracy'],
        'f1_macro': f1_score(true_labels, true_preds, labels=label_list, average='macro', zero_division=0),
    }
    for lbl in label_list:
        metrics[f'f1_{lbl}'] = report[lbl]['f1-score']
        metrics[f'precision_{lbl}'] = report[lbl]['precision']
        metrics[f'recall_{lbl}'] = report[lbl]['recall']
    return metrics


In [ ]:
CHECKPOINT_DIR = f'{DRIVE_ROOT}/SI26-Week7-checkpoints/code-switching-langid'
# ^ deliberately OUTSIDE REPO_ROOT — see note at top of notebook

training_args = TrainingArguments(
    output_dir=CHECKPOINT_DIR,
    num_train_epochs=8,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    greater_is_better=True,
    report_to='none',
    fp16=torch.cuda.is_available(),
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    processing_class=tokenizer,
    data_collator=DataCollatorForTokenClassification(tokenizer),
    compute_metrics=compute_metrics,
)

# Resume from the latest Drive checkpoint if one exists (e.g. after a Colab disconnect)
last_checkpoint = None
if os.path.isdir(CHECKPOINT_DIR):
    existing = [d for d in os.listdir(CHECKPOINT_DIR) if d.startswith('checkpoint-')]
    if existing:
        existing.sort(key=lambda x: int(x.split('-')[-1]))
        last_checkpoint = os.path.join(CHECKPOINT_DIR, existing[-1])

if last_checkpoint:
    print(f'Found existing checkpoint in Drive — resuming from: {last_checkpoint}')
else:
    print('No existing checkpoint found in Drive — starting fresh')

print('Starting training...')
trainer.train(resume_from_checkpoint=last_checkpoint)
print('Training complete!')


## Step 4 — Evaluate: F1 for URD, ENG, MIX

Saved into `WEEK7_DIR` (inside the git repo, not the checkpoint folder) so it's small enough to commit and
paste into your Classroom submission comment.

In [ ]:
eval_results = trainer.evaluate()

print('Final evaluation metrics:')
for k, v in sorted(eval_results.items()):
    if k.startswith('eval_'):
        print(f'  {k[5:]:20s}: {v:.4f}')

with open(f'{WEEK7_DIR}/eval_metrics.json', 'w') as f:
    json.dump(eval_results, f, indent=2)

print(f'\nSaved {WEEK7_DIR}/eval_metrics.json')
print('\nCopy this into your submission comment:')
print(f"  F1 (URD): {eval_results['eval_f1_URD']:.3f}")
print(f"  F1 (ENG): {eval_results['eval_f1_ENG']:.3f}")
print(f"  F1 (MIX): {eval_results['eval_f1_MIX']:.3f}")
print(f"  Macro F1: {eval_results['eval_f1_macro']:.3f}")


> **Heads up — `MIX`'s F1 will come out as `0.000`, and that's expected, not a bug.** I checked your actual
> `dataset.csv`: it's 2,490 word rows across 220 sentences, and every single one is labeled `URD` or `ENG` —
> there are **zero `MIX` labels** in the file. Tracing it back to your Week 6 notebook, the candidate filter
> in Cell 8 requires every word to match `^[A-Za-z']+$`, which has no hyphen in the character class. Since
> `MIX` only ever gets assigned to hyphenated hybrids like `type-kiya` in `label_word()`, any sentence
> containing one gets dropped by `all(re.match(...))` before labeling even happens — so `MIX` candidates were
> filtered out upstream, not just underrepresented.
>
> This doesn't break anything below: `zero_division=0` keeps `compute_metrics` from erroring, you'll just see
> `f1_MIX: 0.0` in the results, which is an honest reflection of the data, not a training failure. For your
> submission comment, it's worth stating this plainly (e.g. "MIX F1 is 0 because the current dataset contains
> no MIX-labeled examples — this traces back to the Week 6 word-filter regex excluding hyphenated tokens").
> If you have time before Friday, the actual fix is a one-line change back in Week 6 Cell 8 —
> `r"^[A-Za-z'-]+$"` (add `-` to the character class) — then re-running Week 6 to regenerate `dataset.csv`
> with real MIX examples before you re-run this notebook.

## Step 5 — Save and push to the Hugging Face Hub

Tip: instead of pasting your token every session, add it once as a Colab secret (key icon in the left
sidebar → "HF_TOKEN") and `notebook_login()` will pick it up automatically — or skip `notebook_login()`
entirely and call `login(token=userdata.get('HF_TOKEN'))` from `google.colab.userdata`.

In [ ]:
from huggingface_hub import notebook_login

notebook_login()  # paste your Hugging Face token (from https://huggingface.co/settings/tokens)


In [ ]:
HF_USERNAME = 'hamnaheh'  # same account used to publish the Week 6 dataset
MODEL_REPO_NAME = 'code-switching-langid-si26-humna'
repo_id = f'{HF_USERNAME}/{MODEL_REPO_NAME}'

model.push_to_hub(repo_id)
tokenizer.push_to_hub(repo_id)

print(f'Model published at: https://huggingface.co/{repo_id}')


## Step 6 — Streamlit demo

Writes `app.py` and `requirements.txt` straight into `WEEK7_DIR` (the git repo in Drive), pointed at the
`repo_id` you just pushed to — no separate hardcoded username to keep in sync.

In [ ]:
APP_PY_TEMPLATE = '''import streamlit as st
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification

MODEL_REPO = "__MODEL_REPO__"

st.set_page_config(page_title="Roman Urdu Code-Switching Language ID", page_icon="\U0001F524")

@st.cache_resource
def load_model():
    tok = AutoTokenizer.from_pretrained(MODEL_REPO)
    mdl = AutoModelForTokenClassification.from_pretrained(MODEL_REPO)
    mdl.eval()
    return tok, mdl

tokenizer, model = load_model()

COLORS = {"URD": "#2ecc71", "ENG": "#3498db", "MIX": "#e67e22"}

def predict(sentence):
    words = sentence.strip().split()
    if not words:
        return []
    encoded = tokenizer(words, is_split_into_words=True, return_tensors="pt", truncation=True)
    with torch.no_grad():
        logits = model(**encoded).logits
    preds = torch.argmax(logits, dim=2)[0].tolist()
    word_ids = encoded.word_ids(batch_index=0)

    results, seen = [], set()
    for idx, wid in enumerate(word_ids):
        if wid is None or wid in seen:
            continue
        seen.add(wid)
        results.append((words[wid], model.config.id2label[preds[idx]]))
    return results

st.title("\U0001F524 Roman Urdu Code-Switching Language ID")
st.caption("Code Saviours SI-26 \u00b7 Project 2 \u00b7 XLM-RoBERTa fine-tuned for token classification")
st.write("Type a Roman Urdu / English mixed sentence \u2014 each word gets tagged URD, ENG, or MIX.")

text = st.text_input("Sentence", "Aaj ka meeting bohot important tha yaar")

if text:
    tagged = predict(text)
    html = " ".join(
        f\'<span style="background-color:{COLORS.get(l, "#bbb")};\'
        f\'padding:2px 6px;border-radius:4px;margin:2px;display:inline-block;">\'
        f\'{w} <sub>{l}</sub></span>\'
        for w, l in tagged
    )
    st.markdown(html, unsafe_allow_html=True)

    st.divider()
    st.subheader("Word-by-word breakdown")
    st.table({"word": [w for w, _ in tagged], "label": [l for _, l in tagged]})
'''

app_py_code = APP_PY_TEMPLATE.replace('__MODEL_REPO__', repo_id)

with open(f'{WEEK7_DIR}/app.py', 'w') as f:
    f.write(app_py_code)

with open(f'{WEEK7_DIR}/requirements.txt', 'w') as f:
    f.write('streamlit\ntransformers\ntorch\nhuggingface_hub\n')

print('Wrote:')
print(f'  {WEEK7_DIR}/app.py')
print(f'  {WEEK7_DIR}/requirements.txt')


> Testing Streamlit apps live inside Colab needs an extra tunneling step (e.g. `pyngrok`), which is more
> setup than it's worth here — it's faster to just deploy (next step) and let Hugging Face Spaces / Streamlit
> Cloud build and serve it for you.

## Step 7 — Deploy

Two easy options — pick whichever's faster for you, both are free:

**Option A — Hugging Face Spaces (matches the handout's "HuggingFace/streamlit" wording):**
1. Go to https://huggingface.co/new-space
2. Space name: e.g. `code-switching-langid-si26-humna-demo`
3. SDK: **Streamlit**
4. Visibility: **Public**
5. Upload `app.py` and `requirements.txt` from `WEEK7_DIR` in Drive (drag-and-drop, or `git push` if you
   clone the Space repo separately)
6. It builds automatically — your demo link will be `https://huggingface.co/spaces/<username>/<space-name>`

**Option B — Streamlit Community Cloud:**
1. Push `app.py` and `requirements.txt` to your GitHub repo (Step 8 below)
2. Go to https://share.streamlit.io, sign in with GitHub, "New app"
3. Point it at this repo, branch, and `SI26-Week7/app.py` as the entry file
4. It deploys and gives you a `*.streamlit.app` link


## Step 8 — Commit and push (straight from Colab)

Since Drive holds a real git working copy, you can push from right here — no need to switch back to
Codespaces. This only works smoothly if you cloned with a token in Step 0b (public repos need no auth to
push if you're already authenticated another way; private repos need the token).

In [ ]:
%cd {WEEK7_DIR}
!git add app.py requirements.txt eval_metrics.json
!git status


In [ ]:
# If this notebook file (SI26-Week7-Humna.ipynb) is saved inside WEEK7_DIR, add it too:
!git add SI26-Week7-Humna.ipynb 2>/dev/null || echo 'Notebook not found in WEEK7_DIR yet — save it there first, see the note in Step 0b'

!git commit -m "Week 7: fine-tuned XLM-RoBERTa language ID model + Streamlit demo"
!git push


## Submission checklist

Paste these three things in Classroom by Friday:

- [ ] **Hugging Face Model Hub link** — `https://huggingface.co/hamnaheh/code-switching-langid-si26-humna` (from Step 5)
- [ ] **Week 7 notebook on GitHub** — the pushed `SI26-Week7-Humna.ipynb` (from Step 8)
- [ ] **Evaluation scores** — F1 for URD, ENG, MIX from `eval_metrics.json` / the printout in Step 4
- [ ] *(bonus, not required but nice to include)* your live demo link from Step 7
